In [4]:
import pandas as pd

In [ ]:
#1. Load Cleaned Dataset
df = pd.read_csv("../data/mockretaildatacleaned.csv")

print("DATASET INFORMATION:")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\nColumns:")
print(df.columns.tolist())


DATASET INFORMATION
Rows: 5000
Columns: 10

Columns:
['customer_id', 'age', 'income', 'previous_purchases', 'campaign_response', 'customer_tenure_days', 'channel', 'avg_basket_size', 'discount', 'purchase']


In [13]:
#2. Define Causal Variable Roles

IDENTIFIER = "customer_id"          # not a model input — keep for merging later

TREATMENT = "discount"              # T
OUTCOME = "purchase"                # Y

CONFOUNDERS = [                     # X
    "age",
    "income",
    "previous_purchases",
    "campaign_response",
    "customer_tenure_days",
    "avg_basket_size",
]

CHANNEL_VAR = "channel"             # W — categorical, encoded separately


In [14]:
#3. Validate Required Columns Exist


required_columns = [IDENTIFIER, TREATMENT, OUTCOME, CHANNEL_VAR] + CONFOUNDERS
missing_columns = [c for c in required_columns if c not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("\nAll required columns are present.")



All required columns are present.


In [16]:
# 4. Check Missing Values


print("MISSING VALUE CHECK:")

missing_values = df[required_columns].isnull().sum()
print(missing_values)

rows_with_missing = df[required_columns].isnull().any(axis=1).sum()
print(f"\nRows containing missing values: {rows_with_missing}")

MISSING VALUE CHECK:
customer_id             0
discount                0
purchase                0
channel                 0
age                     0
income                  0
previous_purchases      0
campaign_response       0
customer_tenure_days    0
avg_basket_size         0
dtype: int64

Rows containing missing values: 0


In [17]:
#5. Check Data Types

print("DATA TYPE CHECK")
print(df[required_columns].dtypes)

expected_numeric = ["age", "previous_purchases", "customer_tenure_days"]
expected_float = ["income", "campaign_response", "avg_basket_size"]

for col in expected_numeric:
    if not pd.api.types.is_integer_dtype(df[col]):
        print(f"WARNING: expected int dtype for '{col}', got {df[col].dtype}")

for col in expected_float:
    if not pd.api.types.is_float_dtype(df[col]):
        print(f"WARNING: expected float dtype for '{col}', got {df[col].dtype}")

if not pd.api.types.is_object_dtype(df[CHANNEL_VAR]) and not pd.api.types.is_string_dtype(df[CHANNEL_VAR]):
    print(f"WARNING: expected string/object dtype for '{CHANNEL_VAR}', got {df[CHANNEL_VAR].dtype}")


DATA TYPE CHECK
customer_id               int64
discount                  int64
purchase                  int64
channel                     str
age                       int64
income                  float64
previous_purchases        int64
campaign_response       float64
customer_tenure_days      int64
avg_basket_size         float64
dtype: object


In [18]:
# 6. Inspect Treatment Values

print("TREATMENT CHECK — discount:")
discount_values = sorted(df[TREATMENT].unique())
print(f"Unique discount levels: {discount_values}")
print(f"Number of levels: {len(discount_values)}")
print("\nValue counts:")
print(df[TREATMENT].value_counts().sort_index())


TREATMENT CHECK — discount:
Unique discount levels: [np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(25), np.int64(30)]
Number of levels: 7

Value counts:
discount
0     2572
5      224
10     437
15     439
20     684
25     439
30     205
Name: count, dtype: int64


In [20]:
 #7. Inspect Outcome Values

print("OUTCOME CHECK — purchase:")
print(f"Unique purchase values: {sorted(df[OUTCOME].unique())}")
print("\nValue counts:")
print(df[OUTCOME].value_counts())

OUTCOME CHECK — purchase:
Unique purchase values: [np.int64(0), np.int64(1)]

Value counts:
purchase
1    2572
0    2428
Name: count, dtype: int64


In [22]:
# 8. Inspect Channel (W) Values

print("CHANNEL (W) CHECK")

print(f"Unique channel values: {df[CHANNEL_VAR].unique().tolist()}")
print("\nValue counts:")
print(df[CHANNEL_VAR].value_counts())


CHANNEL (W) CHECK
Unique channel values: ['in_store', 'online']

Value counts:
channel
in_store    2735
online      2265
Name: count, dtype: int64


In [25]:
# 9. Column Role Documentation

column_roles = {
    "customer_id": {"role": "Identifier", "notes": "Not used as a model input, kept for merging results"},
    "age": {"role": "Confounder (X)", "notes": "Customer age in years"},
    "income": {"role": "Confounder (X)", "notes": "Customer income, float"},
    "previous_purchases": {"role": "Confounder (X)", "notes": "Count of prior purchases"},
    "campaign_response": {"role": "Confounder (X)", "notes": "Historical campaign response rate/score"},
    "customer_tenure_days": {"role": "Confounder (X)", "notes": "Days since customer acquired"},
    "avg_basket_size": {"role": "Confounder (X)", "notes": "Average basket size, float"},
    "channel": {"role": "W", "notes": "Categorical: in_store / online — needs encoding before modeling"},
    "discount": {"role": "Treatment (T)", "notes": "Discount level: 0,5,10,15,20,25,30 — treatment representation (binary vs multi-level) to be finalized by Dishant"},
    "purchase": {"role": "Outcome (Y)", "notes": "Binary purchase indicator"},
}

roles_df = pd.DataFrame(column_roles).T
roles_df.index.name = "column"
roles_df.to_csv("column_roles.csv")


print("COLUMN ROLES SAVED")
print(roles_df)
print("\nSaved to column_roles.csv")



COLUMN ROLES SAVED
                                role  \
column                                 
customer_id               Identifier   
age                   Confounder (X)   
income                Confounder (X)   
previous_purchases    Confounder (X)   
campaign_response     Confounder (X)   
customer_tenure_days  Confounder (X)   
avg_basket_size       Confounder (X)   
channel                            W   
discount               Treatment (T)   
purchase                 Outcome (Y)   

                                                                  notes  
column                                                                   
customer_id           Not used as a model input, kept for merging re...  
age                                               Customer age in years  
income                                           Customer income, float  
previous_purchases                             Count of prior purchases  
campaign_response               Historical campaign resp